In [1]:
import json
from langchain_core.documents import Document
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma


C:\Users\acer\AppData\Local\Temp\ipykernel_21328\2051315301.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import OllamaEmbeddings


# 1. Load the chunks from Phase 1

In [2]:
print("Loading chunks from data/chunks.json...")
with open("data/chunks.json", "r") as f:
    chunk_data = json.load(f)

Loading chunks from data/chunks.json...


# Reconstruct them into LangChain Document objects

In [3]:
documents = [
    Document(page_content=item["page_content"], metadata=item["metadata"]) 
    for item in chunk_data
]
print(f"Loaded {len(documents)} documents ready for vectorization.")

Loaded 5 documents ready for vectorization.


# 2. Initialize Local Embeddings
# This connects to your local Ollama instance running on default port 11434

In [4]:
print("Initializing Ollama Embeddings (nomic-embed-text)...")
embeddings = OllamaEmbeddings(model="nomic-embed-text")

Initializing Ollama Embeddings (nomic-embed-text)...


C:\Users\acer\AppData\Local\Temp\ipykernel_21328\534030443.py:2: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text")


# 3. Initialize ChromaDB and Upsert Data

In [5]:
persist_directory = "data/chroma_db"
print("Upserting documents into ChromaDB... (this might take a moment depending on your hardware)")
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="code_lens_collection"
)
print(f"Success! Embedded and saved {len(documents)} chunks to {persist_directory}.")

Upserting documents into ChromaDB... (this might take a moment depending on your hardware)
Success! Embedded and saved 5 chunks to data/chroma_db.


# 4. Run a quick test query to ensure it worked!

In [6]:
test_query = "How do I complete a task?"
print(f"\nTesting Retrieval for: '{test_query}'")
results = vectorstore.similarity_search(test_query, k=2)
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(f"Code: {doc.page_content.strip()}")


Testing Retrieval for: 'How do I complete a task?'

--- Result 1 ---
Code: function completeTask(taskIndex) {
  taskComplete[taskIndex] = true;
}

--- Result 2 ---
Code: function logTaskState(taskIndex) {
  const title = taskTitles[taskIndex];
  const complete = taskComplete[taskIndex];
  console.log(`${title} has${complete ? " " : " not "}been completed`);
}
